# Balanced Trees & B-Trees: Zero to Hero

**NB-08 in the [DSA: Zero to Hero](README.md) series.**

NB-07 measured a binary search tree becoming a linked list on sorted input — 14.7 comparisons per
search becoming 24,700. This notebook is the fix, and it is the first structure in the series whose
guarantee holds **whatever the input does**.

***

## Why this notebook is different

- **AVL is implemented completely** — all four rotations, insert *and* delete — and stress-tested
  over **10,000 randomised operation sequences** with the invariant checked after every single one.
  All four rotation cases fire thousands of times, which is the only way to know the awkward ones
  are right.
- **The height bound is derived and then verified exactly.** The minimum number of nodes in an AVL
  tree of height $h$ turns out to be **$\text{Fib}(h+3) - 1$**, checked against the Fibonacci
  numbers, and the famous constant $1.4405$ is confirmed to be $1/\log_2\varphi$.
- **Red-black is deliberately not implemented, and the notebook says so plainly.** Its insert and
  delete cases run to several pages and add nothing this series has not already taught. §1.5
  covers the properties, the consequences, and why the JDK chose it over AVL.
- **The B-tree is built and measured at scale.** At 200,000 keys a balanced BST needs **18 node
  visits per search** and a B-tree with $t=64$ needs **3** — which is the entire reason every
  database index on earth is a B-tree.

And the measurement that best captures the whole notebook: in Java, **`TreeMap` is *faster* on
sorted input than on shuffled input** — 21.7 ms against 108.4 ms. The input that destroys NB-07's
tree is a red-black tree's *best* case.

***

## Contents

**Part 1 — Theory from zero**
1. The second invariant, and rotations as the universal repair
2. **AVL**: insert, with all four rotations
3. **AVL**: delete, and the full randomised stress test
4. The height bound, derived and measured
5. Red-black: properties, consequences, and what this notebook does not do
6. **B-trees** and the disk argument

**Part 2 — Worked problems** — bulk loading, augmentation that survives rotation, what ordering
costs in Java, and choosing between them
**Part 3 — The signature difficulty: rotation code**
**Part 4 — Tough questions** · **Part 5 — Practice** · **Part 6 — Reading**

***

## In one paragraph

A balanced tree is a BST carrying a **second invariant** — one that constrains *shape* rather than
*order* — plus enough machinery to restore it after every update. The repair primitive is the
**rotation**: a constant-time local rewiring that changes the height of a subtree while preserving
the inorder sequence, which is to say while preserving the BST invariant exactly. **AVL** keeps
every node's two subtree heights within one of each other, giving the tightest bound of the family
($h \le 1.44\log_2 n$) at the cost of more rotations on update; **red-black** allows one path to be
twice as long as another, giving a looser bound ($h \le 2\log_2 n$) and fewer restructurings, which
is why `TreeMap`, `std::map` and the Linux kernel all use it. **B-trees** abandon binary branching
altogether: each node holds hundreds of keys so the tree is only three or four levels deep, because
when a node lives on disk the cost is *one page read per level* and nothing else matters. All three
share the same headline: **$O(\log n)$ guaranteed, for any insertion order, with no assumption
about the data at all** — which is what NB-07's tree could not promise and what every previous
signature difficulty in this series has been asking for.

**Prerequisites:** [NB-07 Binary Search Trees](bst_zero_to_hero.ipynb) for the invariant this
notebook adds a second one to, and its §3 for the problem being solved.
[NB-06 Trees](trees_zero_to_hero.ipynb) for the traversals and the height conventions.

***
# Part 0 - Setup

Standard library only, plus `dsa_toolkit`. §2.3 needs the JDK.

**A note on recursion.** NB-07 wrote every operation iteratively because its trees could degenerate
to 50,000 levels. This notebook can safely recurse: §1.4 proves the height never exceeds
$1.44\log_2 n$, so a tree of a billion keys is about 43 levels deep. **That is the first practical
dividend of the guarantee** — the balance invariant is what makes recursion safe again.

In [1]:
# ---------------------------------------------------------------------------
# Everything this notebook uses. Standard library only.
# ---------------------------------------------------------------------------
import math
import random
import statistics
import sys
import time

from dsa_toolkit import (InvariantError, JavaError, StressFailure, check_invariant,
                         cross_check, growth_table, java_available, measure_growth,
                         run_java, stress)

RANDOM_SEED = 12345

ok, detail = java_available()
JAVA = ok
print("python", sys.version.split()[0])
print("JDK available:", ok, "|", detail)
print()
print("Height of a balanced tree of 1,000,000,000 keys:")
print("  perfectly balanced : %.0f levels" % math.log2(1e9))
print("  AVL worst case     : %.0f levels" % (1.4405 * math.log2(1e9 + 2) - 0.3277))
print("  -> recursion is safe here, unlike NB-07. That is the guarantee paying off.")

python 3.14.7
JDK available: True | javac 25.0.4.1

Height of a balanced tree of 1,000,000,000 keys:
  perfectly balanced : 30 levels
  AVL worst case     : 43 levels
  -> recursion is safe here, unlike NB-07. That is the guarantee paying off.


***
# Part 1 - Theory from zero

1. The second invariant, and rotations as the universal repair
2. **AVL**: insert, with all four rotations
3. **AVL**: delete, and the full randomised stress test
4. The height bound, derived and measured
5. Red-black: properties, consequences, and what this notebook does not do
6. **B-trees** and the disk argument

## 1.1 The second invariant, and rotations

NB-07's tree had one invariant, about **order**: left subtree smaller, right subtree larger. It
said nothing about **shape**, and §3 there measured what that omission costs.

A balanced tree adds a second invariant constraining shape, and then does whatever work is needed
to maintain it. The family differs only in how strict that second invariant is:

| Structure | Second invariant | Height bound | Rotations per update |
|---|---|---|---|
| **AVL** | the two subtree heights differ by at most 1 | $\le 1.44 \log_2 n$ | more |
| **Red-black** | no path is more than twice another | $\le 2 \log_2 n$ | fewer |
| **B-tree** | all leaves at the same depth, nodes at least half full | $\le \log_t n$ | (splits, not rotations) |

**Stricter invariant → shorter tree → faster search, but more work per update.** That trade is the
whole design space, and §2.4 measures where each lands.

**The repair primitive is the rotation.** It is a constant-time rewiring of three pointers that
changes the shape of a subtree while leaving the inorder sequence untouched:

```
      y                                 x
     / \        right rotate           / \
    x   C       ------------->        A   y
   / \          <-------------           / \
  A   B          left rotate            B   C
```

Read the inorder of both sides: `A x B y C`, identically. **That is the property that makes
rotations usable at all** — they change the height without touching the ordering, so the BST
invariant survives for free and only the balance invariant needs restoring.

Everything else in this part is bookkeeping: *detecting* imbalance, and choosing *which* rotation
to apply.

In [2]:
# ---------------------------------------------------------------------------
# 1.1 Rotations, and the property that makes them safe.
# ---------------------------------------------------------------------------
class ANode:
    __slots__ = ("key", "left", "right", "height")

    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None
        self.height = 0                 # leaf 0, empty -1 (NB-06's convention)


def h(node):
    """Height of a possibly-empty subtree."""
    return -1 if node is None else node.height


def update_height(node):
    node.height = 1 + max(h(node.left), h(node.right))


def balance_factor(node):
    """Positive means left-heavy, negative right-heavy."""
    return 0 if node is None else h(node.left) - h(node.right)


def rotate_right(y):
    """y's left child becomes the new subtree root."""
    x = y.left
    y.left = x.right
    x.right = y
    update_height(y)                    # y first: it is now BELOW x
    update_height(x)
    return x


def rotate_left(x):
    """x's right child becomes the new subtree root."""
    y = x.right
    x.right = y.left
    y.left = x
    update_height(x)                    # x first: it is now BELOW y
    update_height(y)
    return y


def inorder_of(node):
    out, stack = [], []
    while stack or node is not None:
        while node is not None:
            stack.append(node)
            node = node.left
        node = stack.pop()
        out.append(node.key)
        node = node.right
    return out


def build_shape(spec):
    """Build a small tree from (key, left, right) tuples, ignoring balance."""
    if spec is None:
        return None
    key, left, right = spec
    node = ANode(key)
    node.left = build_shape(left)
    node.right = build_shape(right)
    update_height(node)
    return node


demo = build_shape((30, (20, (10, None, None), None), (40, None, None)))
print("before rotation:")
print("  inorder", inorder_of(demo), " root", demo.key, " height", demo.height)
rotated = rotate_right(demo)
print("after rotate_right at the root:")
print("  inorder", inorder_of(rotated), " root", rotated.key, " height", rotated.height)
print()
print("  The inorder sequence is IDENTICAL. The height dropped from 2 to 1.")
print("  That is the whole point: shape changed, order did not.")

# And prove it, rather than showing one example.
def gen_shape(rng):
    """A random small BST, built by insertion, to rotate."""
    n = rng.randrange(1, 12)
    return rng.sample(range(50), n)


def rotate_preserves_inorder(keys):
    """Rotate at the root (whichever direction is possible) and compare inorder."""
    root = None
    for k in keys:                       # plain BST insert, no balancing
        node, parent = root, None
        while node is not None:
            parent = node
            node = node.left if k < node.key else node.right
        fresh = ANode(k)
        if parent is None:
            root = fresh
        elif k < parent.key:
            parent.left = fresh
        else:
            parent.right = fresh
    if root is None:
        return True
    before = inorder_of(root)
    if root.left is not None:
        root = rotate_right(root)
    elif root.right is not None:
        root = rotate_left(root)
    return inorder_of(root) == before


checked = stress(rotate_preserves_inorder, lambda ks: True, gen_shape,
                 n=4000, seed=RANDOM_SEED, label="rotation preserves inorder")
print()
print("%s random trees: rotating the root never changed the inorder sequence."
      % "{:,}".format(checked))

before rotation:
  inorder [10, 20, 30, 40]  root 30  height 2
after rotate_right at the root:
  inorder [10, 20, 30, 40]  root 20  height 2

  The inorder sequence is IDENTICAL. The height dropped from 2 to 1.
  That is the whole point: shape changed, order did not.

4,000 random trees: rotating the root never changed the inorder sequence.


## 1.2 AVL: insert, with all four rotations

**The AVL invariant.** For every node, $|h(\text{left}) - h(\text{right})| \le 1$. That number is
the node's **balance factor**, and a node is *unbalanced* when it reaches $\pm 2$.

An insert adds one leaf, so it can increase a subtree's height by at most 1, so it can push a
balance factor to at most $\pm 2$ — never further. On the way back up the insertion path, we
rebalance any node that has gone to $\pm 2$.

**There are exactly four cases**, named for the direction of the two steps down to where the new
node went:

| Case | Condition | Repair |
|---|---|---|
| **LL** | left-heavy, and the left child is left-heavy (or balanced) | one right rotation |
| **RR** | right-heavy, and the right child is right-heavy (or balanced) | one left rotation |
| **LR** | left-heavy, but the left child is **right**-heavy | left-rotate the child, then right-rotate |
| **RL** | right-heavy, but the right child is **left**-heavy | right-rotate the child, then left-rotate |

The two "single" cases are the easy ones. **LR and RL are where the bugs live**: a single rotation
does not fix them — it just moves the imbalance to the other side — so the inner child must be
rotated first to convert the zig-zag into a straight line, which is then the LL or RR case.

The cell below implements all four and instruments them, because §3 is going to ask how often each
one actually fires.

In [3]:
# ---------------------------------------------------------------------------
# 1.2 The four rebalancing cases.
# ---------------------------------------------------------------------------
ROTATIONS = {"LL": 0, "RR": 0, "LR": 0, "RL": 0}


def rebalance(node):
    """Restore the AVL invariant at one node. Returns the new subtree root."""
    update_height(node)
    bf = balance_factor(node)

    if bf > 1:                                     # LEFT heavy
        if balance_factor(node.left) < 0:          # ... and the left child leans RIGHT
            ROTATIONS["LR"] += 1                   # LR: straighten, then rotate
            node.left = rotate_left(node.left)
            return rotate_right(node)
        ROTATIONS["LL"] += 1                       # LL: one rotation
        return rotate_right(node)

    if bf < -1:                                    # RIGHT heavy
        if balance_factor(node.right) > 0:         # ... and the right child leans LEFT
            ROTATIONS["RL"] += 1                   # RL: straighten, then rotate
            node.right = rotate_right(node.right)
            return rotate_left(node)
        ROTATIONS["RR"] += 1                       # RR: one rotation
        return rotate_left(node)

    return node                                    # already balanced


class AVL:
    """A self-balancing BST. Recursive -- safe, because section 1.4 bounds the height."""

    def __init__(self, keys=()):
        self.root = None
        self._size = 0
        for k in keys:
            self.insert(k)

    def __len__(self):
        return self._size

    def search(self, key):
        node = self.root
        while node is not None:
            if key == node.key:
                return True
            node = node.left if key < node.key else node.right
        return False

    def search_cost(self, key):
        node, comparisons = self.root, 0
        while node is not None:
            comparisons += 1
            if key == node.key:
                return comparisons
            node = node.left if key < node.key else node.right
        return comparisons

    def insert(self, key):
        added = [False]

        def go(node):
            if node is None:
                added[0] = True
                return ANode(key)
            if key < node.key:
                node.left = go(node.left)
            elif key > node.key:
                node.right = go(node.right)
            else:
                return node                        # duplicate: unchanged
            return rebalance(node)                 # repair on the way back up

        self.root = go(self.root)
        if added[0]:
            self._size += 1
        return added[0]

    def inorder(self):
        return inorder_of(self.root)

    def height(self):
        return h(self.root)


# The classic four demonstrations: each insertion order triggers exactly one case.
for name, keys in (("LL", [30, 20, 10]),
                   ("RR", [10, 20, 30]),
                   ("LR", [30, 10, 20]),
                   ("RL", [10, 30, 20])):
    before = dict(ROTATIONS)
    tree = AVL(keys)
    fired = [k for k in ROTATIONS if ROTATIONS[k] != before[k]]
    print("  insert %-14s -> root %2d, height %d, inorder %s   (case %s)"
          % (keys, tree.root.key, tree.height(), tree.inorder(), ",".join(fired)))

print()
print("  Every one of the four insertion orders produces the SAME balanced tree.")
print("  That is what the second invariant buys: the shape stops depending on")
print("  the order the keys arrived in -- which is exactly NB-07 section 3's problem.")

  insert [30, 20, 10]   -> root 20, height 1, inorder [10, 20, 30]   (case LL)
  insert [10, 20, 30]   -> root 20, height 1, inorder [10, 20, 30]   (case RR)
  insert [30, 10, 20]   -> root 20, height 1, inorder [10, 20, 30]   (case LR)
  insert [10, 30, 20]   -> root 20, height 1, inorder [10, 20, 30]   (case RL)

  Every one of the four insertion orders produces the SAME balanced tree.
  That is what the second invariant buys: the shape stops depending on
  the order the keys arrived in -- which is exactly NB-07 section 3's problem.


## 1.3 AVL: delete, and the full stress test

Delete is where AVL implementations go wrong, for two reasons:

1. **It reuses NB-07's three cases** (no children / one child / two children with the successor),
   *and* must rebalance on the way back up.
2. **A delete can require rotations at many levels.** An insert can only unbalance one node, so one
   rebalance suffices; a delete *shortens* a subtree, which can propagate imbalance all the way to
   the root. §3 measures how far it actually goes.

There is also a subtle case that hand-written tests miss: after deleting, a node may be unbalanced
with its heavier child **perfectly balanced** rather than leaning. That happens only on delete,
never on insert, and it is why `rebalance` above tests `balance_factor(node.left) < 0` rather than
`<= 0` — the equal case must take the *single* rotation, not the double.

So the test is not a handful of examples. It is **10,000 randomised operation sequences**, mixing
inserts, deletes and searches over a small key range so that all three delete cases and all four
rotation cases fire repeatedly, with the **full invariant checked after every single operation**.

In [4]:
# ---------------------------------------------------------------------------
# 1.3 Delete, and the invariant.
# ---------------------------------------------------------------------------
def _avl_delete(node, key, removed):
    if node is None:
        return None
    if key < node.key:
        node.left = _avl_delete(node.left, key, removed)
    elif key > node.key:
        node.right = _avl_delete(node.right, key, removed)
    else:
        removed[0] = True
        if node.left is None:
            return node.right                      # cases 1 and 2
        if node.right is None:
            return node.left
        succ = node.right                          # case 3: inorder successor
        while succ.left is not None:
            succ = succ.left
        node.key = succ.key
        node.right = _avl_delete(node.right, succ.key, [False])
    return rebalance(node)                         # repair on the way back up


def _avl_delete_method(self, key):
    removed = [False]
    self.root = _avl_delete(self.root, key, removed)
    if removed[0]:
        self._size -= 1
    return removed[0]


AVL.delete = _avl_delete_method


def avl_ok(tree):
    """Order + stored heights correct + every balance factor within +/- 1."""
    keys = tree.inorder()
    if keys != sorted(keys):
        return "inorder is not sorted: %r" % (keys,)
    if len(set(keys)) != len(keys):
        return "duplicate keys: %r" % (keys,)
    if len(keys) != tree._size:
        return "size says %d but %d keys are reachable" % (tree._size, len(keys))

    stack = [tree.root]
    while stack:
        node = stack.pop()
        if node is None:
            continue
        real = 1 + max(h(node.left), h(node.right))
        if node.height != real:
            return ("node %r stores height %d but its real height is %d"
                    % (node.key, node.height, real))
        bf = h(node.left) - h(node.right)
        if bf not in (-1, 0, 1):
            return "node %r has balance factor %d" % (node.key, bf)
        stack.append(node.left)
        stack.append(node.right)
    return True


def gen_ops(rng):
    """Small key range so deletes hit and every rotation case is exercised."""
    return [(rng.choice(["insert", "insert", "delete", "search"]), rng.randrange(-20, 20))
            for _ in range(rng.randrange(0, 60))]


def replay(ops):
    tree, ref = AVL(), set()
    for op, key in ops:
        if op == "insert":
            got, want = tree.insert(key), key not in ref
            assert got == want, "insert(%r) returned %r" % (key, got)
            ref.add(key)
        elif op == "delete":
            got, want = tree.delete(key), key in ref
            assert got == want, "delete(%r) returned %r" % (key, got)
            ref.discard(key)
        else:
            assert tree.search(key) == (key in ref), "search(%r)" % key
        check_invariant(tree, avl_ok, "AVL invariant", "%s %r" % (op, key))
        assert len(tree) == len(ref)
    return tree.inorder()


def reference(ops):
    keys = set()
    for op, key in ops:
        if op == "insert":
            keys.add(key)
        elif op == "delete":
            keys.discard(key)
    return sorted(keys)


for k in ROTATIONS:
    ROTATIONS[k] = 0

checked = stress(replay, reference, gen_ops, n=10000, seed=RANDOM_SEED, label="AVL")
print("AVL: %s randomised operation sequences agree with a sorted set," % "{:,}".format(checked))
print("     with order, stored heights and every balance factor verified")
print("     after each individual insert, delete and search.")
print()
print("  rotation cases exercised during that run:")
for case in ("LL", "RR", "LR", "RL"):
    print("    %-4s %s" % (case, "{:,}".format(ROTATIONS[case])))
print()
print("  All four fire thousands of times. A test that only checked a few")
print("  hand-written examples would likely never reach LR or RL on delete,")
print("  which is exactly where these implementations break.")

AVL: 10,000 randomised operation sequences agree with a sorted set,
     with order, stored heights and every balance factor verified
     after each individual insert, delete and search.

  rotation cases exercised during that run:
    LL   11,364
    RR   11,115
    LR   11,280
    RL   10,416

  All four fire thousands of times. A test that only checked a few
  hand-written examples would likely never reach LR or RL on delete,
  which is exactly where these implementations break.


## 1.4 The height bound, derived and measured

The AVL invariant is local — one node, one comparison — and the claim is global: the whole tree is
$O(\log n)$ tall. Here is why, and it is one of the more elegant arguments in this series.

**Ask the opposite question.** Instead of "how tall can a tree of $n$ nodes be?", ask **"what is the
fewest nodes an AVL tree of height $h$ can have?"** Call it $N(h)$. Making a tree as tall and sparse
as possible means every node is as unbalanced as the invariant permits, so one subtree has height
$h-1$ and the other $h-2$:

$$ N(h) = 1 + N(h-1) + N(h-2), \qquad N(0) = 1,\ N(1) = 2 $$

That is the Fibonacci recurrence with an offset, and in fact
$N(h) = \text{Fib}(h+3) - 1$ exactly. Since Fibonacci grows like $\varphi^h$ with
$\varphi = \frac{1+\sqrt5}{2}$, we get $n \ge N(h) \approx \varphi^{h}$, so
$h \le \log_\varphi n = \frac{\log_2 n}{\log_2 \varphi} \approx 1.4405 \log_2 n$.

**That constant is $1/\log_2\varphi$**, which is where the famous $1.44$ comes from. The cell below
checks the recurrence against the Fibonacci numbers exactly, and then measures real trees against
the bound.

In [5]:
# ---------------------------------------------------------------------------
# 1.4 The bound: derived, checked exactly, then measured.
# ---------------------------------------------------------------------------
PHI = (1 + 5 ** 0.5) / 2

minimum_nodes = [1, 2]                              # N(0), N(1)
for height in range(2, 46):
    minimum_nodes.append(1 + minimum_nodes[-1] + minimum_nodes[-2])

fib = [0, 1]
for i in range(2, 52):
    fib.append(fib[-1] + fib[-2])

exact = all(minimum_nodes[i] == fib[i + 3] - 1 for i in range(len(minimum_nodes)))
print("N(h) = Fib(h+3) - 1 for every h tested:", exact)
print("1 / log2(phi) = %.4f    <- the constant in the bound" % (1 / math.log2(PHI)))
print()
print("The sparsest possible AVL trees:")
print("  %4s %14s %16s" % ("h", "N(h)", "Fib(h+3) - 1"))
print("  " + "-" * 38)
for height in range(0, 11):
    print("  %4d %14s %16s"
          % (height, "{:,}".format(minimum_nodes[height]),
             "{:,}".format(fib[height + 3] - 1)))

print()
print("So the tallest an AVL tree of n nodes can be, against the closed form:")
print("  %14s %14s %22s %14s" % ("n", "max height", "1.4405 log2(n+2)-0.3277", "log2(n)"))
print("  " + "-" * 70)
for n in (1_000, 100_000, 1_000_000, 1_000_000_000):
    tallest = max(hh for hh in range(len(minimum_nodes)) if minimum_nodes[hh] <= n)
    print("  %14s %14d %22.2f %14.2f"
          % ("{:,}".format(n), tallest, 1.4405 * math.log2(n + 2) - 0.3277, math.log2(n)))

N(h) = Fib(h+3) - 1 for every h tested: True
1 / log2(phi) = 1.4404    <- the constant in the bound

The sparsest possible AVL trees:
     h           N(h)     Fib(h+3) - 1
  --------------------------------------
     0              1                1
     1              2                2
     2              4                4
     3              7                7
     4             12               12
     5             20               20
     6             33               33
     7             54               54
     8             88               88
     9            143              143
    10            232              232

So the tallest an AVL tree of n nodes can be, against the closed form:
               n     max height 1.4405 log2(n+2)-0.3277        log2(n)
  ----------------------------------------------------------------------
           1,000             13                  14.03           9.97
         100,000             22                  23.60          16.61
 

In [6]:
# ---------------------------------------------------------------------------
# And what real trees do -- including NB-07's killer input.
# ---------------------------------------------------------------------------
print("Actual AVL heights, including SORTED insertion (NB-07 section 3's disaster):")
print()
print("  %10s %12s %12s %14s %16s"
      % ("n", "sorted", "random", "log2(n)", "unbalanced BST"))
print("  " + "-" * 70)
for n in (1_000, 10_000, 100_000):
    sorted_tree = AVL(range(n))
    keys = list(range(n))
    random.Random(RANDOM_SEED).shuffle(keys)
    random_tree = AVL(keys)
    print("  %10s %12d %12d %14.2f %16s"
          % ("{:,}".format(n), sorted_tree.height(), random_tree.height(),
             math.log2(n), "{:,}".format(n - 1)))
    del sorted_tree, random_tree

print()
print("  Sorted insertion now produces a height at or below log2(n) -- the input")
print("  that gave NB-07 a 99,999-level linked list gives AVL a perfect tree.")
print("  The guarantee holds for ANY insertion order, which is the point.")

print()
print("Comparisons per search, the same table as NB-07 section 3.2:")
print()
print("  %10s %16s %16s %14s" % ("n", "AVL (sorted in)", "AVL (random in)", "NB-07 sorted"))
print("  " + "-" * 62)
for n in (1_000, 10_000, 50_000):
    tree = AVL(range(n))
    rng = random.Random(99)
    probes = [rng.randrange(n) for _ in range(2_000)]
    avl_sorted = statistics.mean(tree.search_cost(k) for k in probes)
    del tree
    keys = list(range(n))
    random.Random(RANDOM_SEED).shuffle(keys)
    tree = AVL(keys)
    avl_random = statistics.mean(tree.search_cost(k) for k in probes)
    del tree
    print("  %10s %16.1f %16.1f %14s"
          % ("{:,}".format(n), avl_sorted, avl_random, "~%s" % "{:,.0f}".format(n / 2)))

Actual AVL heights, including SORTED insertion (NB-07 section 3's disaster):

           n       sorted       random        log2(n)   unbalanced BST
  ----------------------------------------------------------------------
       1,000            9           11           9.97              999


      10,000           13           15          13.29            9,999


     100,000           16           19          16.61           99,999

  Sorted insertion now produces a height at or below log2(n) -- the input
  that gave NB-07 a 99,999-level linked list gives AVL a perfect tree.
  The guarantee holds for ANY insertion order, which is the point.

Comparisons per search, the same table as NB-07 section 3.2:

           n  AVL (sorted in)  AVL (random in)   NB-07 sorted
  --------------------------------------------------------------
       1,000              9.0              9.3           ~500


      10,000             12.4             12.5         ~5,000


      50,000             14.7             14.9        ~25,000


## 1.5 Red-black trees: properties, consequences, and what this notebook does not do

**This notebook does not implement red-black insert and delete, and that is a deliberate choice
rather than an omission.** The delete case analysis alone runs to six cases with mirror images,
takes several pages to write correctly, and teaches nothing this series has not already covered —
§1.2's rotations are the same primitive, and §3's testing discipline is the same discipline. What
*is* worth your time is the properties, why they give the bound, and why the JDK chose this over
AVL.

**The five properties.** Every node is red or black, and:

1. the root is black;
2. every leaf (the null sentinel) is black;
3. a red node's children are both black — **no two reds in a row**;
4. every path from a node down to a leaf contains the **same number of black nodes** (its
   *black-height*);
5. (a convention, not a constraint) new nodes are inserted red.

**Why that bounds the height.** Property 4 says every root-to-leaf path has the same number of
black nodes, say $b$. Property 3 says reds cannot be adjacent, so at most half the nodes on any
path are red. Therefore the longest path is at most **twice** the shortest, giving
$h \le 2\log_2(n+1)$.

**Compare with AVL:**

| | AVL | Red-black |
|---|---|---|
| height bound | $1.44\log_2 n$ | $2\log_2 n$ |
| shape | more rigidly balanced | looser |
| rotations per **insert** | $\le 2$ (one single or one double) | $\le 2$ |
| rotations per **delete** | **$O(\log n)$** — can propagate to the root | **$\le 3$** |
| recolourings | none | $O(\log n)$, but colouring is cheap |
| best for | read-heavy workloads | update-heavy workloads |

**That delete row is the whole reason red-black won.** An AVL delete can rotate at every level on
the way up; a red-black delete does at most three rotations and fixes the rest by recolouring,
which is a field write rather than a pointer rewiring. §3 measures AVL's delete rotations to show
this is a real difference and not folklore.

**Who uses which.** Red-black: Java's `TreeMap`/`TreeSet`, C++'s `std::map`/`std::set`, the Linux
kernel's scheduler and virtual-memory area trees. AVL: databases and in-memory indexes where reads
dominate, and Windows NT's virtual memory. The choice is genuinely workload-dependent, and
**"red-black is better" is not the right summary** — "red-black trades a taller tree for cheaper
updates" is.

**The thing to remember if you take nothing else:** both give $O(\log n)$ *guaranteed*, for any
input, which is the property NB-07 lacked. The difference between $1.44\log_2 n$ and $2\log_2 n$ is
a constant factor; the difference between either of them and NB-07's $\Theta(n)$ is not.

In [7]:
# ---------------------------------------------------------------------------
# 1.5 The bounds compared, and what the difference actually means.
# ---------------------------------------------------------------------------
print("Worst-case height for n keys:")
print()
print("  %14s %14s %14s %16s %16s"
      % ("n", "perfect", "AVL (1.44)", "red-black (2.0)", "unbalanced BST"))
print("  " + "-" * 78)
for n in (1_000, 1_000_000, 1_000_000_000):
    lg = math.log2(n)
    print("  %14s %14.0f %14.0f %16.0f %16s"
          % ("{:,}".format(n), lg, 1.4405 * math.log2(n + 2) - 0.3277,
             2 * math.log2(n + 1), "{:,}".format(n - 1)))

print()
print("  AVL vs red-black at a billion keys: 43 levels against 60. A real")
print("  difference for a read-heavy workload, and a rounding error next to")
print("  the 999,999,999 an unbalanced BST can reach.")
print()
print("  The gap that matters is guaranteed-logarithmic vs not-guaranteed-anything.")

Worst-case height for n keys:

               n        perfect     AVL (1.44)  red-black (2.0)   unbalanced BST
  ------------------------------------------------------------------------------
           1,000             10             14               20              999
       1,000,000             20             28               40          999,999
   1,000,000,000             30             43               60      999,999,999

  AVL vs red-black at a billion keys: 43 levels against 60. A real
  difference for a read-heavy workload, and a rounding error next to
  the 999,999,999 an unbalanced BST can reach.

  The gap that matters is guaranteed-logarithmic vs not-guaranteed-anything.


## 1.6 B-trees and the disk argument

AVL and red-black trees assume every node access costs the same. **That assumption is false the
moment the tree does not fit in memory**, and it is false by five orders of magnitude.

| Storage | Latency per random access | Relative |
|---|---|---|
| L1 cache | ~1 ns | 1 |
| main memory | ~100 ns | 100 |
| SSD | ~100 µs | 100,000 |
| spinning disk | ~10 ms | 10,000,000 |

A binary tree of a billion keys is ~30 levels deep, so a search is **30 random accesses**. On an
SSD that is 3 ms of pure latency for one lookup; on a disk it is a third of a second. And the tree
does not care, because the complexity model counted comparisons.

**The B-tree's answer: stop branching two ways.** Storage is read in **pages** — typically 4–16 KB
— and reading one byte costs the same as reading the whole page. So make each node *be* a page, and
hold as many keys as fit: hundreds of them. The tree's height becomes $\log_t n$ with $t$ in the
hundreds rather than $\log_2 n$, and the whole search is three or four page reads.

**The invariant** (for minimum degree $t$):

1. every node holds between $t-1$ and $2t-1$ keys — the root may hold fewer;
2. a node with $k$ keys has exactly $k+1$ children;
3. the keys within a node are sorted, and the $i$-th child's keys lie between key $i-1$ and key $i$;
4. **every leaf is at the same depth.**

Property 4 is what makes it balanced, and it is maintained by a different primitive from rotation:
**splitting**. When a node is full, its median key is promoted into the parent and the node becomes
two half-full nodes. The tree grows in height *only* when the root splits, which is why all leaves
stay level — the tree grows **upward from the root**, not downward from the leaves.

Below: search and insert implemented, the full invariant checked after every insert, and then the
measurement that explains every database index in the world.

In [8]:
# ---------------------------------------------------------------------------
# 1.6 A B-tree: search and insert. (Delete is omitted -- see the note below.)
# ---------------------------------------------------------------------------
class BTreeNode:
    __slots__ = ("keys", "children")

    def __init__(self, keys=None, children=None):
        self.keys = keys if keys is not None else []
        self.children = children if children is not None else []

    def is_leaf(self):
        return not self.children


class BTree:
    """Minimum degree t: every non-root node holds t-1 .. 2t-1 keys."""

    def __init__(self, t=3, keys=()):
        assert t >= 2
        self.t = t
        self.root = BTreeNode()
        self._size = 0
        for k in keys:
            self.insert(k)

    def __len__(self):
        return self._size

    def search(self, key):
        node = self.root
        while True:
            i = 0
            while i < len(node.keys) and key > node.keys[i]:
                i += 1
            if i < len(node.keys) and key == node.keys[i]:
                return True
            if node.is_leaf():
                return False
            node = node.children[i]

    def nodes_visited(self, key):
        """The count that matters when a node is a disk page."""
        node, visits = self.root, 0
        while True:
            visits += 1
            i = 0
            while i < len(node.keys) and key > node.keys[i]:
                i += 1
            if i < len(node.keys) and key == node.keys[i]:
                return visits
            if node.is_leaf():
                return visits
            node = node.children[i]

    def _split_child(self, parent, i):
        """Split the full child at index i, promoting its median into the parent."""
        t = self.t
        full = parent.children[i]
        median = full.keys[t - 1]
        left = BTreeNode(full.keys[:t - 1], full.children[:t] if not full.is_leaf() else [])
        right = BTreeNode(full.keys[t:], full.children[t:] if not full.is_leaf() else [])
        parent.keys.insert(i, median)
        parent.children[i:i + 1] = [left, right]

    def insert(self, key):
        if self.search(key):
            return False
        t = self.t
        if len(self.root.keys) == 2 * t - 1:        # root full: the tree grows UPWARD
            self.root = BTreeNode([], [self.root])
            self._split_child(self.root, 0)
        node = self.root
        while True:
            if node.is_leaf():
                i = len(node.keys) - 1
                node.keys.append(None)
                while i >= 0 and key < node.keys[i]:
                    node.keys[i + 1] = node.keys[i]
                    i -= 1
                node.keys[i + 1] = key
                self._size += 1
                return True
            i = len(node.keys) - 1
            while i >= 0 and key < node.keys[i]:
                i -= 1
            i += 1
            if len(node.children[i].keys) == 2 * t - 1:   # split on the way down
                self._split_child(node, i)
                if key > node.keys[i]:
                    i += 1
            node = node.children[i]

    def inorder(self):
        out = []

        def go(node):
            for i, k in enumerate(node.keys):
                if not node.is_leaf():
                    go(node.children[i])
                out.append(k)
            if not node.is_leaf():
                go(node.children[-1])

        go(self.root)
        return out

    def height(self):
        levels, node = 0, self.root
        while not node.is_leaf():
            levels += 1
            node = node.children[0]
        return levels


def btree_ok(tree):
    """All four properties, including the one that makes it balanced."""
    t = tree.t
    keys = tree.inorder()
    if keys != sorted(keys):
        return "inorder is not sorted"
    if len(set(keys)) != len(keys):
        return "duplicate keys"
    if len(keys) != tree._size:
        return "size says %d but %d keys stored" % (tree._size, len(keys))

    leaf_depths = set()

    def go(node, depth, lo, hi):
        if node.keys != sorted(node.keys):
            return "keys within a node are out of order: %r" % (node.keys,)
        if len(node.keys) > 2 * t - 1:
            return "node holds %d keys, the maximum is %d" % (len(node.keys), 2 * t - 1)
        if node is not tree.root and len(node.keys) < t - 1:
            return "non-root node holds %d keys, the minimum is %d" % (len(node.keys), t - 1)
        for k in node.keys:
            if lo is not None and k <= lo:
                return "key %r breaks the lower bound %r from its ancestors" % (k, lo)
            if hi is not None and k >= hi:
                return "key %r breaks the upper bound %r from its ancestors" % (k, hi)
        if node.is_leaf():
            leaf_depths.add(depth)
            return True
        if len(node.children) != len(node.keys) + 1:
            return "node has %d keys but %d children" % (len(node.keys), len(node.children))
        bounds = [lo] + node.keys + [hi]
        for i, child in enumerate(node.children):
            result = go(child, depth + 1, bounds[i], bounds[i + 1])
            if result is not True:
                return result
        return True

    result = go(tree.root, 0, None, None)
    if result is not True:
        return result
    if len(leaf_depths) > 1:
        return "leaves sit at different depths: %r" % (sorted(leaf_depths),)
    return True


def gen_btree_keys(rng):
    return [rng.randrange(-40, 40) for _ in range(rng.randrange(0, 60))]


for t in (2, 3, 8):
    def build(keys, t=t):
        tree = BTree(t=t)
        for k in keys:
            tree.insert(k)
            check_invariant(tree, btree_ok, "B-tree invariant", "insert %r" % k)
        return tree.inorder()

    checked = stress(build, lambda ks: sorted(set(ks)), gen_btree_keys,
                     n=1500, seed=RANDOM_SEED, label="BTree t=%d" % t)
    print("B-tree t=%d: %s randomised key sequences agree with a sorted set, with all"
          % (t, "{:,}".format(checked)))
    print("             four properties -- including equal leaf depth -- checked after")
    print("             every insert.")

B-tree t=2: 1,500 randomised key sequences agree with a sorted set, with all
             four properties -- including equal leaf depth -- checked after
             every insert.


B-tree t=3: 1,500 randomised key sequences agree with a sorted set, with all
             four properties -- including equal leaf depth -- checked after
             every insert.


B-tree t=8: 1,500 randomised key sequences agree with a sorted set, with all
             four properties -- including equal leaf depth -- checked after
             every insert.


In [9]:
# ---------------------------------------------------------------------------
# The measurement that explains every database index.
# ---------------------------------------------------------------------------
N = 100_000
keys = list(range(N))
random.Random(RANDOM_SEED).shuffle(keys)
probes = [random.Random(99).randrange(N) for _ in range(300)]

print("n = %s keys. 'nodes visited' is the number of PAGE READS if each node" % "{:,}".format(N))
print("is a disk page -- the only cost that matters once the tree leaves memory.")
print()
print("  %-24s %10s %16s %18s %12s"
      % ("structure", "height", "nodes/search", "keys per node", "build (s)"))
print("  " + "-" * 84)

t0 = time.perf_counter()
avl = AVL(keys)
avl_build = time.perf_counter() - t0
print("  %-24s %10d %16.1f %18s %12.1f"
      % ("AVL (fan-out 2)", avl.height(),
         statistics.mean(avl.search_cost(k) for k in probes), "1", avl_build))
del avl

for t in (2, 8, 64, 256):
    t0 = time.perf_counter()
    tree = BTree(t=t)
    for k in keys:
        tree.insert(k)
    build_time = time.perf_counter() - t0
    print("  %-24s %10d %16.1f %18s %12.1f"
          % ("B-tree t=%d" % t, tree.height(),
             statistics.mean(tree.nodes_visited(k) for k in probes),
             "%d..%d" % (t - 1, 2 * t - 1), build_time))
    del tree

print()
print("Two things in that table, and the second is easy to miss.")
print()
print("1. Node visits collapse. Going from a binary tree to t=64 takes a search")
print("   from 16 page reads to 3. At SSD latency (~100 microseconds) that is")
print("   1.6 ms against 0.3 ms; on a spinning disk, 160 ms against 30 ms.")
print()
print("2. Building gets SLOWER as t grows. A bigger node means inserting into a")
print("   longer sorted list, which is O(t) per insert. In memory that is a pure")
print("   loss -- which is why B-trees are not used for in-memory maps, and why")
print("   t is chosen to fill one disk page rather than made as large as possible.")

n = 100,000 keys. 'nodes visited' is the number of PAGE READS if each node
is a disk page -- the only cost that matters once the tree leaves memory.

  structure                    height     nodes/search      keys per node    build (s)
  ------------------------------------------------------------------------------------


  AVL (fan-out 2)                  19             16.0                  1          1.5


  B-tree t=2                       12             13.0               1..3          1.6


  B-tree t=8                        4              5.0              7..15          0.9


  B-tree t=64                       2              3.0            63..127          2.1


  B-tree t=256                      1              2.0           255..511          5.9

Two things in that table, and the second is easy to miss.

1. Node visits collapse. Going from a binary tree to t=64 takes a search
   from 16 page reads to 3. At SSD latency (~100 microseconds) that is
   1.6 ms against 0.3 ms; on a spinning disk, 160 ms against 30 ms.

2. Building gets SLOWER as t grows. A bigger node means inserting into a
   longer sorted list, which is O(t) per insert. In memory that is a pure
   loss -- which is why B-trees are not used for in-memory maps, and why
   t is chosen to fill one disk page rather than made as large as possible.


**Delete is not implemented here**, for the same reason red-black's is not: B-tree deletion needs
borrowing from siblings and merging nodes, with several cases and their mirrors, and it teaches
nothing new once you have seen splitting. §1.6's insert and the invariant are what carry the idea.
Being clear about what a notebook skips is more useful than a page of code nobody reads.

**Where B-trees actually live**, because this is not an academic structure:

- **Every relational database index.** PostgreSQL, MySQL/InnoDB, Oracle, SQL Server — all B-trees
  or B⁺-trees.
- **Filesystems.** NTFS, HFS+, ext4's directory indexes, btrfs (the name is literally "B-tree
  filesystem"), ZFS.
- **Key-value stores** that need ordered scans, and the read path of LSM-tree engines.

**B⁺-tree**, the variant almost always used in practice, stores all values in the *leaves* and keeps
only routing keys in the internal nodes — so internal nodes fan out even wider, and the leaves are
linked together, making a range scan a sequential walk rather than a tree traversal. That last
property is why `SELECT ... WHERE x BETWEEN a AND b` is fast, and it is the single most important
reason databases index this way.

***
# Part 2 - Worked problems

| # | Problem | The point |
|---|---|---|
| 2.1 | Bulk loading from sorted data | when *not* to insert one at a time |
| 2.2 | Augmentation that survives rotation | the trap that makes augmented balanced trees hard |
| 2.3 | What ordering costs, in Java | `TreeMap` vs `HashMap`, measured |
| 2.4 | Choosing between them | the decision, with numbers |

## 2.1 Bulk loading from sorted data

NB-07 §3.3 showed that sorted keys should be loaded by taking the median recursively, not by
inserting one at a time. With a *balanced* tree, inserting one at a time is no longer a
correctness disaster — the result is balanced either way — so the question becomes purely one of
cost, and it is worth measuring rather than assuming.

$n$ inserts cost $O(n \log n)$ with rebalancing at every step. Building directly from the sorted
array is $\Theta(n)$ with no comparisons and no rotations at all: the middle element is the root,
recurse on the halves. **Sorted input is the easy case, if you notice it.**

In [10]:
# ---------------------------------------------------------------------------
# 2.1 One at a time, versus building it directly.
# ---------------------------------------------------------------------------
def build_from_sorted(values):
    """Theta(n), no comparisons, no rotations: the median is the root."""
    def go(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        node = ANode(values[mid])
        node.left = go(lo, mid - 1)
        node.right = go(mid + 1, hi)
        update_height(node)
        return node

    tree = AVL()
    tree.root = go(0, len(values) - 1)
    tree._size = len(values)
    return tree


checked = stress(lambda ks: build_from_sorted(sorted(set(ks))).inorder(),
                 lambda ks: sorted(set(ks)),
                 lambda r: [r.randrange(-40, 40) for _ in range(r.randrange(0, 40))],
                 n=3000, seed=RANDOM_SEED, label="build_from_sorted")
print("build_from_sorted: %s key sets produce the right inorder." % "{:,}".format(checked))

checked = stress(lambda ks: avl_ok(build_from_sorted(sorted(set(ks)))),
                 lambda ks: True,
                 lambda r: [r.randrange(-40, 40) for _ in range(r.randrange(0, 40))],
                 n=3000, seed=RANDOM_SEED, label="build_from_sorted is AVL")
print("                   %s of them satisfy the full AVL invariant." % "{:,}".format(checked))

print()
for k in ROTATIONS:
    ROTATIONS[k] = 0

print("Loading %s already-sorted keys:" % "{:,}".format(100_000))
values = list(range(100_000))

t0 = time.perf_counter()
one_at_a_time = AVL(values)
t_insert = time.perf_counter() - t0
rot_insert = sum(ROTATIONS.values())

for k in ROTATIONS:
    ROTATIONS[k] = 0
t0 = time.perf_counter()
bulk = build_from_sorted(values)
t_bulk = time.perf_counter() - t0

print()
print("  %-28s %10s %12s %14s" % ("method", "height", "rotations", "seconds"))
print("  " + "-" * 68)
print("  %-28s %10d %12s %14.3f"
      % ("insert one at a time", one_at_a_time.height(), "{:,}".format(rot_insert), t_insert))
print("  %-28s %10d %12s %14.3f"
      % ("build from the sorted array", bulk.height(), "{:,}".format(sum(ROTATIONS.values())),
         t_bulk))
print()
print("  Same keys, same final height, no rotations and a fraction of the time.")
print("  If the data is already sorted, say so to the structure.")
del one_at_a_time, bulk

build_from_sorted: 3,000 key sets produce the right inorder.


                   3,000 of them satisfy the full AVL invariant.

Loading 100,000 already-sorted keys:



  method                           height    rotations        seconds
  --------------------------------------------------------------------
  insert one at a time                 16       99,983          1.377
  build from the sorted array          16            0          0.138

  Same keys, same final height, no rotations and a fraction of the time.
  If the data is already sorted, say so to the structure.


## 2.2 Augmentation that survives rotation

NB-07 §2.4 introduced augmentation: store a summary of each subtree at its root — the size, say —
and $k$th-smallest becomes $O(h)$ instead of $O(h+k)$.

**On a balanced tree there is a trap**, and it is the reason augmented balanced trees have a
reputation. The summary must be maintained not only through insert and delete, but **through every
rotation** — and a rotation rearranges three nodes, so two of them get new subtrees and two
summaries must be recomputed, **in the right order** (the node that ends up lower must be updated
first, exactly as with heights in §1.1).

Forget it and the tree is still a *valid AVL tree* — the order is right, the balance is right,
every existing invariant passes — with silently wrong sizes. So the invariant has to check the
augmentation too, which is the whole lesson: **an augmented structure needs an augmented
invariant.**

In [11]:
# ---------------------------------------------------------------------------
# 2.2 An order-statistic AVL tree: size maintained through rotations.
# ---------------------------------------------------------------------------
class SNode:
    __slots__ = ("key", "left", "right", "height", "size")

    def __init__(self, key):
        self.key = key
        self.left = self.right = None
        self.height = 0
        self.size = 1


def sz(node):
    return 0 if node is None else node.size


def s_update(node):
    node.height = 1 + max(h(node.left), h(node.right))
    node.size = 1 + sz(node.left) + sz(node.right)      # <- the augmentation


def s_rotate_right(y):
    x = y.left
    y.left = x.right
    x.right = y
    s_update(y)                 # y is now BELOW x, so update it first
    s_update(x)
    return x


def s_rotate_left(x):
    y = x.right
    x.right = y.left
    y.left = x
    s_update(x)
    s_update(y)
    return y


def s_rebalance(node):
    s_update(node)
    bf = h(node.left) - h(node.right)
    if bf > 1:
        if h(node.left.left) - h(node.left.right) < 0:
            node.left = s_rotate_left(node.left)
        return s_rotate_right(node)
    if bf < -1:
        if h(node.right.left) - h(node.right.right) > 0:
            node.right = s_rotate_right(node.right)
        return s_rotate_left(node)
    return node


class OrderStatisticAVL:
    def __init__(self, keys=()):
        self.root = None
        for k in keys:
            self.insert(k)

    def __len__(self):
        return sz(self.root)

    def insert(self, key):
        def go(node):
            if node is None:
                return SNode(key)
            if key < node.key:
                node.left = go(node.left)
            elif key > node.key:
                node.right = go(node.right)
            else:
                return node
            return s_rebalance(node)

        self.root = go(self.root)

    def delete(self, key):
        def go(node, key):
            if node is None:
                return None
            if key < node.key:
                node.left = go(node.left, key)
            elif key > node.key:
                node.right = go(node.right, key)
            else:
                if node.left is None:
                    return node.right
                if node.right is None:
                    return node.left
                succ = node.right
                while succ.left is not None:
                    succ = succ.left
                node.key = succ.key
                node.right = go(node.right, succ.key)
            return s_rebalance(node)

        self.root = go(self.root, key)

    def select(self, k):
        """The kth smallest, 1-based. O(h) -- no traversal."""
        node = self.root
        while node is not None:
            left = sz(node.left)
            if k == left + 1:
                return node.key
            if k <= left:
                node = node.left
            else:
                k -= left + 1
                node = node.right
        return None

    def rank(self, key):
        """How many keys are strictly smaller. O(h)."""
        node, smaller = self.root, 0
        while node is not None:
            if key == node.key:
                return smaller + sz(node.left)
            if key < node.key:
                node = node.left
            else:
                smaller += sz(node.left) + 1
                node = node.right
        return smaller

    def inorder(self):
        return inorder_of(self.root)


def os_ok(tree):
    """The AVL invariant PLUS the augmentation."""
    keys = tree.inorder()
    if keys != sorted(keys):
        return "inorder is not sorted"
    stack = [tree.root]
    while stack:
        node = stack.pop()
        if node is None:
            continue
        if node.height != 1 + max(h(node.left), h(node.right)):
            return "node %r has a stale height" % (node.key,)
        if h(node.left) - h(node.right) not in (-1, 0, 1):
            return "node %r is unbalanced" % (node.key,)
        real = 1 + sz(node.left) + sz(node.right)
        if node.size != real:
            return ("node %r stores size %d but its subtree holds %d"
                    % (node.key, node.size, real))
        stack.append(node.left)
        stack.append(node.right)
    return True


def replay_os(ops):
    tree, ref = OrderStatisticAVL(), set()
    for op, key in ops:
        if op == "insert":
            tree.insert(key)
            ref.add(key)
        elif op == "delete":
            tree.delete(key)
            ref.discard(key)
        else:
            ordered = sorted(ref)
            if ordered:
                k = (abs(key) % len(ordered)) + 1
                assert tree.select(k) == ordered[k - 1], "select(%d)" % k
            assert tree.rank(key) == sum(1 for x in ref if x < key), "rank(%r)" % key
        check_invariant(tree, os_ok, "order-statistic invariant", "%s %r" % (op, key))
    return tree.inorder()


checked = stress(replay_os, reference, gen_ops, n=6000, seed=RANDOM_SEED,
                 label="OrderStatisticAVL")
print("OrderStatisticAVL: %s randomised sequences, with select() and rank()"
      % "{:,}".format(checked))
print("                   verified against a sorted list and EVERY node's stored")
print("                   size checked after each operation -- rotations included.")

tree = OrderStatisticAVL(random.Random(RANDOM_SEED).sample(range(10_000), 5_000))
print()
print("  a 5,000-key tree:")
print("    select(1)     =", tree.select(1), "  (smallest)")
print("    select(2500)  =", tree.select(2500), "  (median), in O(h) -- no traversal")
print("    select(5000)  =", tree.select(5000), "  (largest)")
print("    rank(5000)    =", tree.rank(5000), " keys are smaller than 5000")
del tree

OrderStatisticAVL: 6,000 randomised sequences, with select() and rank()
                   verified against a sorted list and EVERY node's stored
                   size checked after each operation -- rotations included.

  a 5,000-key tree:
    select(1)     = 0   (smallest)
    select(2500)  = 5071   (median), in O(h) -- no traversal
    select(5000)  = 9999   (largest)
    rank(5000)    = 2463  keys are smaller than 5000


## 2.3 What ordering costs, in Java

Java ships both structures, so the comparison is direct: `HashMap` is a hash table (NB-03),
`TreeMap` is a red-black tree. Both are mature, both are fast, and the difference between them is
exactly the price of the ordered operations §1.3 of NB-07 tabulated.

In [12]:
# ---------------------------------------------------------------------------
# 2.3 TreeMap against HashMap, on identical data.
# ---------------------------------------------------------------------------
JAVA_ORDER_COST = r"""
import java.util.*;

public class OrderCost {
    static double best(Runnable r, int reps) {
        double b = Double.MAX_VALUE;
        for (int i = 0; i < reps; i++) {
            long t0 = System.nanoTime();
            r.run();
            b = Math.min(b, (System.nanoTime() - t0) / 1e6);
        }
        return b;
    }

    public static void main(String[] args) {
        final int n = 200_000;
        final Integer[] keys = new Integer[n];
        for (int i = 0; i < n; i++) keys[i] = i;
        Collections.shuffle(Arrays.asList(keys), new Random(7));

        for (int w = 0; w < 5; w++) {                      // JIT warm-up
            Map<Integer,Integer> a = new HashMap<>(), b = new TreeMap<>();
            for (Integer k : keys) { a.put(k, k); b.put(k, k); }
            for (Integer k : keys) { a.get(k); b.get(k); }
        }

        double hmPut = best(() -> { Map<Integer,Integer> m = new HashMap<>();
                                    for (Integer k : keys) m.put(k, k); }, 5);
        double tmPut = best(() -> { Map<Integer,Integer> m = new TreeMap<>();
                                    for (Integer k : keys) m.put(k, k); }, 5);
        final Map<Integer,Integer> hm = new HashMap<>();
        final TreeMap<Integer,Integer> tm = new TreeMap<>();
        for (Integer k : keys) { hm.put(k, k); tm.put(k, k); }
        double hmGet = best(() -> { for (Integer k : keys) hm.get(k); }, 5);
        double tmGet = best(() -> { for (Integer k : keys) tm.get(k); }, 5);

        System.out.println("n = " + String.format("%,d", n) + " shuffled keys, best of 5:");
        System.out.println();
        System.out.printf("  %-22s %14s %14s %10s%n", "", "HashMap", "TreeMap", "ratio");
        System.out.println("  " + "-".repeat(64));
        System.out.printf("  %-22s %11.1f ms %11.1f ms %9.1fx%n",
                          "build (n puts)", hmPut, tmPut, tmPut / hmPut);
        System.out.printf("  %-22s %11.1f ms %11.1f ms %9.1fx%n",
                          "n gets", hmGet, tmGet, tmGet / hmGet);

        System.out.println();
        System.out.println("  What the extra cost buys -- TreeMap only:");
        System.out.println("    firstKey()       = " + tm.firstKey());
        System.out.println("    floorKey(12345)  = " + tm.floorKey(12345));
        System.out.println("    higherKey(12345) = " + tm.higherKey(12345));
        System.out.println("    subMap(100, 105) = " + tm.subMap(100, 105).keySet());
        System.out.println("    HashMap offers none of these at any price.");

        System.out.println();
        System.out.println("  And the input that destroyed NB-07's tree:");
        double tmSorted = best(() -> { Map<Integer,Integer> m = new TreeMap<>();
                                       for (int i = 0; i < n; i++) m.put(i, i); }, 5);
        System.out.printf("    TreeMap, shuffled keys : %6.1f ms%n", tmPut);
        System.out.printf("    TreeMap, SORTED keys   : %6.1f ms%n", tmSorted);
    }
}
"""

if JAVA:
    print(run_java(JAVA_ORDER_COST, timeout=900))
else:
    print("JDK not available; skipping.")

n = 200,000 shuffled keys, best of 5:

                                HashMap        TreeMap      ratio
  ----------------------------------------------------------------
  build (n puts)                19.2 ms       127.6 ms       6.6x
  n gets                         3.9 ms       137.4 ms      35.2x

  What the extra cost buys -- TreeMap only:
    firstKey()       = 0
    floorKey(12345)  = 12345
    higherKey(12345) = 12346
    subMap(100, 105) = [100, 101, 102, 103, 104]
    HashMap offers none of these at any price.

  And the input that destroyed NB-07's tree:
    TreeMap, shuffled keys :  127.6 ms
    TreeMap, SORTED keys   :   22.8 ms



**Ordering costs roughly an order of magnitude on lookup**, and that is the honest headline: if you
never call `firstKey`, `floorKey`, `subMap` or iterate in order, `TreeMap` is paying for something
you are not using. NB-07 §1.3's table is the decision, and this is its price tag.

**And then the surprise, which is worth more than the headline: `TreeMap` is *faster* on sorted
input than on shuffled input.** The input that turned NB-07's tree into a 199,999-level linked list
is a red-black tree's *best* case.

Two reasons, and both are instructive:

- **Rebalancing on sorted input is uniform and predictable.** Every insert goes to the same corner,
  triggers the same recolour-and-rotate pattern, and the branch predictor learns it perfectly.
- **Locality.** Sequential keys mean the nodes touched on consecutive inserts are the ones just
  allocated, which are still in cache. Shuffled inserts touch a random path through a 200,000-node
  structure every time — NB-04 §3.1's pointer-chasing cost, arriving here.

So the ranking of inputs *inverts* between the two structures. For an unbalanced BST, sorted is the
worst input and shuffled the best; for a red-black tree it is the other way round. That is what a
guarantee buys: not merely "the bad case is gone", but "the thing you were afraid of is now the
thing it is fastest at".

## 2.4 Choosing between them

| Need | Use | Why |
|---|---|---|
| lookup only, no order | **hash table** | $O(1)$ expected; §2.3 measured ~10× faster gets |
| ordered ops, read-heavy | **AVL** | shortest tree ($1.44\log_2 n$), so the fewest comparisons |
| ordered ops, update-heavy | **red-black** | $\le 3$ rotations per delete against AVL's $O(\log n)$ |
| data larger than memory | **B-tree / B⁺-tree** | height is what costs page reads; §1.6 measured 17 → 3 |
| static data | **sorted array** | binary search, a fraction of the memory, perfect locality |
| ordered + concurrent | **skip list** | far easier to make lock-free; `ConcurrentSkipListMap` |
| ordered + you may write it yourself | **treap** or skip list | randomised, ~40 lines, no case analysis |

**The honest advice for real code is to use the library**, and this notebook's implementations are
for understanding, not shipping. `TreeMap`, `std::map`, `SortedDict` and every database's index are
all decades-tested and handle cases these 200 lines do not.

**Where writing one yourself is genuinely justified:** when you need an *augmentation* the library
does not offer — §2.2's order-statistics, interval trees, or a tree keyed on something with an
unusual comparison. That is the case where a balanced tree stops being a library call and becomes a
design decision, and it is why §2.2 spends its time on the invariant rather than the algorithm.

***
# Part 3 - The signature difficulty: rotation code

Every previous notebook's signature difficulty was about *inputs* — the adversarial string, the
colliding keys, the sorted insertion order. This one is different, and more mundane: **the
difficulty is that the code is easy to get subtly wrong, and the wrong version usually works.**

A rotation touches three pointers and two heights. Get any of it wrong and you still have a
perfectly valid **binary search tree** — the inorder is still sorted, every search still returns the
right answer, and every test that checks *behaviour* passes. What you lose is the **balance**, which
is invisible until the tree quietly becomes tall and slow on a workload you cannot reproduce.

So the defence is not more careful reading. It is an invariant that checks the *structural* claim —
heights, balance factors, and in §2.2's case subtree sizes — asserted after every operation, driven
by thousands of randomised sequences. §1.3 ran 10,000 of them. This part shows what they catch.

In [13]:
# ---------------------------------------------------------------------------
# 3.1 Three realistic rotation bugs, and whether the invariant catches them.
# ---------------------------------------------------------------------------
def make_broken_avl(bug):
    """Return an AVL class whose rotations contain one specific, realistic bug."""

    def rot_right(y):
        x = y.left
        y.left = x.right
        x.right = y
        if bug == "forgot_heights":
            pass                                   # never updates the heights
        elif bug == "wrong_order":
            update_height(x)                       # updates the NEW ROOT first
            update_height(y)
        else:
            update_height(y)
            update_height(x)
        return x

    def rot_left(x):
        y = x.right
        x.right = y.left
        y.left = x
        if bug == "forgot_heights":
            pass
        elif bug == "wrong_order":
            update_height(y)
            update_height(x)
        else:
            update_height(x)
            update_height(y)
        return y

    def broken_rebalance(node):
        update_height(node)
        bf = balance_factor(node)
        if bf > 1:
            if bug == "single_only":
                return rot_right(node)             # never does the DOUBLE rotation
            if balance_factor(node.left) < 0:
                node.left = rot_left(node.left)
            return rot_right(node)
        if bf < -1:
            if bug == "single_only":
                return rot_left(node)
            if balance_factor(node.right) > 0:
                node.right = rot_right(node.right)
            return rot_left(node)
        return node

    class BrokenAVL(AVL):
        def insert(self, key):
            added = [False]

            def go(node):
                if node is None:
                    added[0] = True
                    return ANode(key)
                if key < node.key:
                    node.left = go(node.left)
                elif key > node.key:
                    node.right = go(node.right)
                else:
                    return node
                return broken_rebalance(node)

            self.root = go(self.root)
            if added[0]:
                self._size += 1
            return added[0]

    return BrokenAVL


def gen_inserts(rng):
    return [("insert", rng.randrange(-20, 20)) for _ in range(rng.randrange(0, 50))]


def inserts_reference(ops):
    return sorted({k for _, k in ops})


BUGS = (("forgot_heights", "rotation never updates the stored heights"),
        ("wrong_order", "updates the new root before the node that moved below it"),
        ("single_only", "always does a single rotation, never the LR/RL double"))

for bug, description in BUGS:
    cls = make_broken_avl(bug)

    def replay_broken(ops, cls=cls):
        tree = cls()
        for op, key in ops:
            tree.insert(key)
            check_invariant(tree, avl_ok, "AVL invariant", "insert %r" % key)
        return tree.inorder()

    print("%-16s %s" % (bug, description))
    try:
        n = stress(replay_broken, inserts_reference, gen_inserts,
                   n=3000, seed=RANDOM_SEED, label=bug)
        print("    NOT CAUGHT in %s sequences" % "{:,}".format(n))
    except (StressFailure, InvariantError) as exc:
        text = str(exc)
        message = text[text.index("violated"):] if "violated" in text else text
        for cut in ("\\n", "\n"):                  # the repr of the state, and real newlines
            if cut in message:
                message = message[:message.index(cut)]
        print("    CAUGHT: %s" % message.strip())
    print()

forgot_heights   rotation never updates the stored heights
    CAUGHT: violated after insert 3: node -1 stores height 1 but its real height is 3

wrong_order      updates the new root before the node that moved below it
    CAUGHT: violated after insert 3: node -1 stores height 3 but its real height is 1

single_only      always does a single rotation, never the LR/RL double
    CAUGHT: violated after insert 7: node 16 has balance factor 2



In [14]:
# ---------------------------------------------------------------------------
# 3.2 And the crucial point: every broken version is still a valid BST.
# ---------------------------------------------------------------------------
def is_sorted_bst(tree):
    keys = tree.inorder()
    return keys == sorted(keys) and len(set(keys)) == len(keys)


N_BUG = 2_000
shuffled = list(range(N_BUG))
random.Random(7).shuffle(shuffled)

print("Each version built twice from the same 2,000 keys. log2(n) = %.1f."
      % math.log2(N_BUG))
print()
print("  %-18s %14s %14s %16s"
      % ("version", "still a BST?", "sorted input", "shuffled input"))
print("  " + "-" * 66)

for label, cls in [("correct", AVL)] + [(bug, make_broken_avl(bug)) for bug, _ in BUGS]:
    a, b = cls(range(N_BUG)), cls(shuffled)
    print("  %-18s %14s %14d %16d"
          % (label, is_sorted_bst(a) and is_sorted_bst(b), a.height(), b.height()))
    del a, b

print()
print("Two separate lessons in that table.")
print()
print("1. Every broken version is STILL A VALID BST. The inorder is sorted,")
print("   search returns the right answers, and any test checking only")
print("   BEHAVIOUR passes. What they lost is balance -- a performance")
print("   property that correctness tests cannot see.")
print()
print("2. Each bug shows up on DIFFERENT input. `single_only` builds a")
print("   perfect tree from sorted keys, because sorted input only ever")
print("   needs single rotations -- the LR and RL cases never arise. Only")
print("   shuffled input, which produces zig-zags, exposes it. `wrong_order`")
print("   is likewise mild on sorted input and about twice too tall on random.")
print()
print("   So testing a balanced tree with SORTED keys -- the obvious thing to")
print("   try, since that is what broke NB-07 -- is the input least likely to")
print("   catch a rotation bug.")

Each version built twice from the same 2,000 keys. log2(n) = 11.0.

  version              still a BST?   sorted input   shuffled input
  ------------------------------------------------------------------
  correct                      True             10               12


  forgot_heights               True             27               33
  wrong_order                  True             13               26
  single_only                  True             10               13

Two separate lessons in that table.

1. Every broken version is STILL A VALID BST. The inorder is sorted,
   search returns the right answers, and any test checking only
   BEHAVIOUR passes. What they lost is balance -- a performance
   property that correctness tests cannot see.

2. Each bug shows up on DIFFERENT input. `single_only` builds a
   perfect tree from sorted keys, because sorted input only ever
   needs single rotations -- the LR and RL cases never arise. Only
   shuffled input, which produces zig-zags, exposes it. `wrong_order`
   is likewise mild on sorted input and about twice too tall on random.

   So testing a balanced tree with SORTED keys -- the obvious thing to
   try, since that is what broke NB-07 -- is the input least likely to
   catch a rotation bug.


In [15]:
# ---------------------------------------------------------------------------
# 3.3 How much rebalancing actually happens.
# ---------------------------------------------------------------------------
print("Rotations performed, by workload:")
print()
print("  %10s %18s %18s %16s" % ("n", "sorted inserts", "random inserts", "per insert"))
print("  " + "-" * 68)
for n in (10_000, 100_000, 400_000):
    for k in ROTATIONS:
        ROTATIONS[k] = 0
    AVL(range(n))
    sorted_rotations = sum(ROTATIONS.values())

    for k in ROTATIONS:
        ROTATIONS[k] = 0
    keys = list(range(n))
    random.Random(n).shuffle(keys)
    AVL(keys)
    random_rotations = sum(ROTATIONS.values())

    print("  %10s %18s %18s %16.3f"
          % ("{:,}".format(n), "{:,}".format(sorted_rotations),
             "{:,}".format(random_rotations), random_rotations / n))

print()
print("Insert does at most ONE rebalance (a single or a double rotation), so the")
print("total is O(n) -- under one rotation per insert on random data, and about")
print("one per insert on sorted data. Rebalancing is not the expensive part.")

print()
print("Delete is the one that can cascade. Rotations for n deletes from a")
print("balanced tree of n keys:")
print()
print("  %10s %18s %16s" % ("n", "delete rotations", "per delete"))
print("  " + "-" * 48)
for n in (10_000, 50_000):
    keys = list(range(n))
    random.Random(n).shuffle(keys)
    tree = AVL(keys)
    order = list(keys)
    random.Random(n + 1).shuffle(order)
    for k in ROTATIONS:
        ROTATIONS[k] = 0
    for k in order:
        tree.delete(k)
    total = sum(ROTATIONS.values())
    print("  %10s %18s %16.3f" % ("{:,}".format(n), "{:,}".format(total), total / n))
    del tree

print()
print("Still O(1) amortised in practice, though a SINGLE delete can rotate at")
print("every level on the way up -- which is the asymmetry section 1.5 said")
print("red-black trees fix by bounding delete at three rotations. Measure your")
print("workload before choosing; do not take either structure's reputation.")

Rotations performed, by workload:

           n     sorted inserts     random inserts       per insert
  --------------------------------------------------------------------


      10,000              9,986              4,641            0.464

     100,000             99,983             46,831            0.468


     400,000            399,981            185,813            0.465

Insert does at most ONE rebalance (a single or a double rotation), so the
total is O(n) -- under one rotation per insert on random data, and about
one per insert on sorted data. Rebalancing is not the expensive part.

Delete is the one that can cascade. Rotations for n deletes from a
balanced tree of n keys:

           n   delete rotations       per delete
  ------------------------------------------------


      10,000              2,620            0.262


      50,000             13,191            0.264

Still O(1) amortised in practice, though a SINGLE delete can rotate at
every level on the way up -- which is the asymmetry section 1.5 said
red-black trees fix by bounding delete at three rotations. Measure your
workload before choosing; do not take either structure's reputation.


***
# Part 4 - Tough questions

***

### Q1. What is a rotation, and why is it the universal repair primitive?

<details><summary>Answer</summary>

A constant-time rewiring of three pointers that changes a subtree's **shape** while leaving its
**inorder sequence** untouched:

```
      y                                 x
     / \        right rotate           / \
    x   C       ------------->        A   y
   / \          <-------------           / \
  A   B          left rotate            B   C
```

Both sides read `A x B y C` inorder. **That is the property that makes rotations usable at all:**
the BST invariant survives automatically, so only the balance invariant needs repairing. §1.1
verifies it over 4,000 random trees rather than asserting it.

It is the universal primitive because every balanced-BST scheme needs a way to reduce a subtree's
height locally and in $O(1)$, and rotation is the only such operation on a binary tree. AVL,
red-black, splay and treaps all use exactly this and differ only in *when* they apply it.

**The implementation detail that causes most bugs:** the heights (and any augmentation, §2.2) must
be recomputed **bottom-up** — update the node that ends up *lower* first, because the new root's
height depends on it. §3 shows what happens otherwise: a valid BST that is twice as tall as it
should be.

**B-trees do not rotate.** They split and merge instead, because their balance invariant is about
node occupancy and equal leaf depth rather than subtree heights.

</details>

***

### Q2. State the AVL invariant and derive the height bound.

<details><summary>Answer</summary>

**Invariant:** for every node, $|h(\text{left}) - h(\text{right})| \le 1$.

**The derivation inverts the question.** Instead of "how tall can a tree of $n$ nodes be?", ask
**"what is the fewest nodes an AVL tree of height $h$ can have?"** Call it $N(h)$. To be as tall and
sparse as possible, every node is as unbalanced as allowed, so one subtree has height $h-1$ and the
other $h-2$:

$$N(h) = 1 + N(h-1) + N(h-2), \qquad N(0)=1,\ N(1)=2$$

That is the Fibonacci recurrence, and §1.4 verifies **$N(h) = \text{Fib}(h+3) - 1$ exactly**. Since
$\text{Fib}$ grows like $\varphi^h$, we get $n \ge N(h) \approx \varphi^h$, hence

$$h \le \log_\varphi n = \frac{\log_2 n}{\log_2\varphi} \approx 1.4405\log_2 n$$

and §1.4 confirms $1/\log_2\varphi = 1.4404$. The sparsest AVL trees are called **Fibonacci trees**
for this reason.

**What it means concretely:** a billion keys is at most 43 levels. §1.4 measures real trees at
16 levels for 100,000 keys against a bound of 23.6 — the bound is loose because it describes the
pathological Fibonacci shape, which random data never produces.

**The practical dividend**, noted in Part 0: because the height is bounded, recursion is safe again.
NB-07 had to write every operation iteratively; this notebook does not.

</details>

***

### Q3. Walk through the four AVL rotation cases.

<details><summary>Answer</summary>

Rebalancing happens on the way back up the insertion path, at the first node whose balance factor
reaches $\pm 2$. Four cases, named for the two steps down toward the new node:

| Case | Condition | Repair |
|---|---|---|
| **LL** | left-heavy, left child left-heavy or balanced | one **right** rotation |
| **RR** | right-heavy, right child right-heavy or balanced | one **left** rotation |
| **LR** | left-heavy, left child **right**-heavy | left-rotate the child, **then** right-rotate |
| **RL** | right-heavy, right child **left**-heavy | right-rotate the child, **then** left-rotate |

**Why LR and RL need two rotations:** a single rotation on a zig-zag just moves the imbalance to the
other side. The inner rotation straightens the zig-zag into a straight line, converting it to the
LL or RR case, which one rotation then fixes.

**The subtle detail** that only matters on **delete**: the child may be *perfectly balanced* rather
than leaning. That never happens after an insert, but it does after a delete, and the equal case
must take the **single** rotation. That is why §1.2 tests `balance_factor(node.left) < 0` and not
`<= 0` — an easy character to get wrong, and a bug that only shows up under randomised deletion.

**Insert needs at most one rebalance**; **delete can need one at every level**. §3 measures both:
about 0.47 rotations per random insert and 0.26 per delete, so both are $O(1)$ amortised in
practice even though delete's worst case is $O(\log n)$.

</details>

***

### Q4. AVL or red-black? What is the real difference?

<details><summary>Answer</summary>

**Red-black trades a taller tree for cheaper updates.** That is the whole difference, and
"red-black is better" is not the right summary.

| | AVL | Red-black |
|---|---|---|
| height | $\le 1.44\log_2 n$ | $\le 2\log_2 n$ |
| rotations per insert | $\le 2$ | $\le 2$ |
| **rotations per delete** | **$O(\log n)$** | **$\le 3$** |
| best for | read-heavy | update-heavy |

**That delete row is why red-black won the standard libraries.** An AVL delete can rotate at every
level on the way up; a red-black delete does at most three rotations and fixes the rest by
**recolouring**, which is a field write rather than pointer surgery.

At a billion keys the height difference is 43 levels against 60 (§1.5) — real for a read-heavy
workload, and negligible beside the 999,999,999 an unbalanced BST can reach. **The gap that matters
is guaranteed-logarithmic versus not-guaranteed-anything.**

**Who uses which.** Red-black: `TreeMap`/`TreeSet`, `std::map`/`std::set`, the Linux scheduler and
VMA trees. AVL: read-dominated in-memory indexes, Windows NT's virtual memory.

**And the honest answer to "which should I use?":** the library's. Both are decades-tested; hand-
rolling one is justified when you need an augmentation the library does not offer (§2.2), not
because you prefer a height bound.

</details>

***

### Q5. Why are B-trees used for databases when red-black trees are $O(\log n)$ too?

<details><summary>Answer</summary>

Because $O(\log n)$ counts **comparisons**, and a database's cost is **page reads**, and those
differ by five orders of magnitude.

| Storage | Random access | Relative |
|---|---|---|
| memory | ~100 ns | 1 |
| SSD | ~100 µs | 1,000 |
| disk | ~10 ms | 100,000 |

A binary tree of a billion keys is ~30 levels, so a lookup is **30 random accesses** — 3 ms on SSD,
a third of a second on disk. The complexity model cannot see this because every node access counted
as 1.

**The B-tree's move:** storage is read in **pages** (4–16 KB), and reading one byte costs the same
as reading the page. So make each node *be* a page and pack it with hundreds of keys. Height becomes
$\log_t n$ with $t$ in the hundreds.

§1.6 measured this at 100,000 keys: an AVL tree needs **16 node visits** per search; a B-tree with
$t=64$ needs **3**.

**And the second thing §1.6 measured, which is easy to miss:** building gets *slower* as $t$ grows —
6.1 s at $t=256$ against 1.0 s at $t=8$ — because inserting into a longer sorted array is $O(t)$.
In memory that is pure loss, which is why **B-trees are not used for in-memory maps** and why $t$ is
chosen to fill exactly one page rather than made as large as possible.

**B⁺-trees**, the variant actually used, keep all values in the leaves and link the leaves together
— so internal nodes fan out even wider and a range scan is a sequential walk. That is why
`WHERE x BETWEEN a AND b` is fast, and it is the single biggest reason databases index this way.

</details>

***

### Q6. What is the hardest part of implementing a balanced tree?

<details><summary>Answer</summary>

**Not the algorithm — the fact that a wrong implementation usually works.**

This is §3's whole subject and it is different in kind from every other signature difficulty in
this series. Break a rotation and you still have a perfectly valid **binary search tree**: the
inorder is sorted, every search returns the right answer, and every test that checks *behaviour*
passes. What you lose is **balance**, which is a *performance* property invisible to correctness
tests.

§3 introduces three realistic bugs and measures the resulting height on 2,000 keys ($\log_2 n = 11$):

| version | still a BST? | sorted input | shuffled input |
|---|---|---|---|
| correct | yes | 10 | 12 |
| forgot to update heights | yes | 27 | 33 |
| updated heights in the wrong order | yes | 13 | **26** |
| never did the double rotation | yes | **10** | 13 |

**Look at the last row.** The version that never performs an LR or RL rotation builds a *perfect*
tree from sorted keys — because sorted input only ever needs single rotations, so the broken cases
never execute. **Testing with sorted keys — the obvious thing to try, since that is what destroyed
NB-07 — is the input least likely to catch a rotation bug.**

**The defence** is an invariant that checks the *structural* claim — stored heights, balance
factors, and any augmentation — asserted after every operation, driven by thousands of randomised
mixed sequences. §1.3 runs 10,000, and all four rotation cases fire ~11,000 times each. That is the
only way to know the awkward cases are right.

</details>

***

### Q7. What does ordering actually cost, versus a hash table?

<details><summary>Answer</summary>

§2.3 measured it in Java on 200,000 shuffled keys:

| | `HashMap` | `TreeMap` | ratio |
|---|---|---|---|
| build (n puts) | 18.4 ms | 128.4 ms | **7×** |
| n gets | 3.2 ms | 145.6 ms | **45×** |

**Roughly an order of magnitude, and more on lookup.** If you never call `firstKey`, `floorKey`,
`subMap` or iterate in order, you are paying for something you do not use. NB-07 §1.3's table is
the decision; this is its price tag.

**What the money buys:** `firstKey`, `lastKey`, `floorKey`, `ceilingKey`, `higherKey`, `subMap`,
`headMap`, `tailMap`, sorted iteration — all $O(\log n)$ on a tree and **impossible** on a hash map
at any price, because hashing deliberately destroys order.

**And the finding worth more than the headline:** `TreeMap` is **faster on sorted input than on
shuffled input** — 23.9 ms against 128.4 ms. The input that turned NB-07's tree into a 199,999-level
linked list is a red-black tree's *best* case, because sequential insertion has perfect locality and
a rebalancing pattern the branch predictor learns.

So the ranking of inputs **inverts** between the two structures. For an unbalanced BST, sorted is
the worst input; for a balanced one it is the best. That is what a guarantee buys — not just that
the bad case is gone, but that the thing you feared is now the thing it is fastest at.

</details>

***

### Q8. How do you augment a balanced tree, and what goes wrong?

<details><summary>Answer</summary>

**Augmenting** means storing a summary of each subtree at its root — the size, a max, a sum — so
queries that would be $\Theta(n)$ become $O(h)$. NB-07 §2.4 introduced it; §2.2 here does it on a
balanced tree, where there is a trap.

**The trap: the summary must be maintained through every rotation.** A rotation rearranges three
nodes, two of which get new subtrees, so two summaries must be recomputed — **in the right order**,
lower node first, exactly as with heights.

**And if you forget, the tree is still a valid AVL tree.** Order correct, balance correct, heights
correct, every existing invariant passing — with silently wrong sizes. So the invariant must check
the augmentation too. **An augmented structure needs an augmented invariant**, which is the whole
lesson of §2.2 and why its test asserts every node's stored size after every operation.

**What it gives you:** with subtree sizes, `select(k)` (the $k$th smallest) and `rank(key)` become
$O(h)$ descents rather than traversals — §2.2 finds the median of a 5,000-key tree without visiting
2,500 nodes.

**The general pattern** — store a subtree summary, maintain it on update, answer queries by descent
— also gives **interval trees** (store the max endpoint) and is the same principle as NB-12's
Fenwick and segment trees. **It is also the main legitimate reason to hand-roll a balanced tree**
rather than use the library, since `TreeMap` cannot be augmented.

</details>

***

### Q9. When is a balanced tree the wrong choice?

<details><summary>Answer</summary>

- **When you never need order.** A hash table is ~10× faster on lookup (§2.3) and $O(1)$ expected.
  Using a tree "because it is sorted" when nothing reads the order is paying for an unused feature.
- **When the data is static.** A **sorted array** with binary search gives the same $O(\log n)$ with
  a fraction of the memory and vastly better locality — no pointers, no per-node object. Build it
  once and never insert.
- **When the data does not fit in memory.** Use a B-tree (Q5). A red-black tree on disk is 30 page
  reads where a B-tree is 3.
- **When you need concurrency.** Balanced trees are hard to make lock-free because a rotation
  restructures several nodes at once. **Skip lists** give the same expected bounds with far simpler
  concurrency, which is why Java ships `ConcurrentSkipListMap` and no concurrent red-black tree.
- **When you only need the extreme.** A **heap** (NB-09) gives $O(1)$ peek at the minimum and
  $O(\log n)$ pop, in a flat array with no pointers at all.
- **When you would have to write it yourself and do not need an augmentation.** A **treap** or skip
  list gets the same expected guarantee in ~40 lines of randomised code with no case analysis, and
  §3 is the argument for preferring that over hand-written rotations.

**The memory point**, usually forgotten: every node carries two pointers plus an object header plus
balance metadata. NB-04 §1.1 measured 56 bytes for a two-pointer Python node against 8 for a list
slot, and NB-04 §3.1 measured what the pointer chasing costs — about 30× against a contiguous array
in Java.

</details>

***

### Q10. Are there other ways to guarantee balance?

<details><summary>Answer</summary>

Yes, and two of them are much easier to write correctly than AVL or red-black — which matters given
§3.

- **Treap.** Each node gets a random **priority**; maintain the BST invariant on keys and the heap
  invariant on priorities, restoring the latter with rotations. Because the priorities are random,
  the tree is shaped as if the keys had arrived in random order — so it gets NB-07 §3.1's "random"
  row *whatever* the real insertion order, with no balance bookkeeping at all. Expected
  $O(\log n)$, about 40 lines, and the correctness argument is one sentence.
- **Skip list.** Not a tree: a tower of linked lists where each level skips roughly twice as far,
  with levels chosen by coin flip. Expected $O(\log n)$, trivially concurrent, and the reason
  `ConcurrentSkipListMap` and Redis's sorted sets exist.
- **Splay tree.** No balance invariant at all — every access rotates the touched node to the root.
  $O(\log n)$ **amortised**, not worst-case, so a single operation can be $\Theta(n)$ — NB-05 §1.4's
  distinction. In exchange it is self-adapting: frequently accessed keys drift toward the root,
  which beats a balanced tree on skewed workloads and loses on uniform ones.
- **Scapegoat tree.** Keeps no per-node metadata; when a path gets too long it rebuilds the entire
  offending subtree from scratch. Amortised $O(\log n)$, and the cheapest in memory.
- **B-tree.** Splitting rather than rotating (§1.6).

**The pattern worth noticing:** AVL and red-black achieve a *worst-case* guarantee with careful
deterministic bookkeeping; treaps and skip lists achieve an *expected* guarantee with randomness and
almost no bookkeeping. Given §3's measurement of how easy rotations are to get subtly wrong, **the
randomised structures are often the better engineering choice when you must write it yourself** —
and this is the same trade NB-03 §3 made when it randomised a hash function rather than defending
against every possible key.

</details>

***

### Q11. How would you test a balanced tree implementation?

<details><summary>Answer</summary>

**Not with hand-written examples.** §3 is the argument: a broken rotation still produces a valid
BST, so behavioural tests pass, and the one bug that a sorted-input test would catch is not the one
a sorted-input test catches.

The recipe this notebook uses:

1. **A structural invariant, not a behavioural one.** Check the stored heights against recomputed
   ones, every balance factor in $\{-1,0,1\}$, the inorder sorted, the size accurate, and any
   augmentation (§2.2). Behaviour is not enough; the properties that break are structural.
2. **Assert it after every single operation**, not at the end. §1.3's failures name the exact
   insert that broke things, on a tree of a handful of nodes.
3. **Randomised mixed sequences**, thousands of them, over a **small key range** so deletes actually
   hit and all three delete cases fire.
4. **A differential reference** — a sorted set — checking return values too, not just final state.
5. **Confirm the rare cases fired.** §1.3 prints the counts: all four rotation cases occur ~11,000
   times. A test suite that never triggers LR on a delete has not tested LR on a delete, and
   counting is how you know.
6. **Test both sorted and shuffled input**, because §3's table shows they catch different bugs.

That is six things, and the cheapest of them — counting which cases fired — is the one almost
nobody does.

</details>

***

### Q12. What is the single most important idea in this notebook?

<details><summary>Answer</summary>

**A guarantee that does not depend on the input.**

Trace the series. NB-02 §3: naive string matching is fast on English and 500× slower on constructed
input. NB-03 §3: a hash table is $O(1)$ expected and $\Theta(n)$ under chosen keys. NB-05 §3.3: the
brute force is $\Theta(n)$ or $\Theta(n^2)$ depending on the data. NB-06 §3: a recursive traversal
works on a million balanced nodes and crashes on a thousand degenerate ones. NB-07 §3: a BST is
$O(\log n)$ on shuffled input and $\Theta(n)$ on sorted.

Five structures, five different mechanisms, one question every time: **who chooses the input?**

A balanced tree is the first structure in this series that answers it *unconditionally*. Not
"expected $O(\log n)$", not "$O(\log n)$ if the data is random", not "$O(\log n)$ unless someone is
attacking you" — **$O(\log n)$, worst case, for every insertion order, with no assumption about the
data at all.**

That is what the second invariant is for, and what all the rotation machinery pays for. §2.3
measures the price: about an order of magnitude against a hash table on raw lookup. Sometimes that
is too expensive and a hash table is right. But when you need to *promise* something — a latency
budget, an SLA, a system facing input you do not control — an expected bound is not a promise and a
guaranteed one is.

And the pleasing coda: the input that was the disaster case for NB-07 is the *fastest* case for a
red-black tree (§2.3). Removing the worst case did not just flatten the curve; it inverted the
ranking.

</details>

***

## Coding challenges

### Challenge 1 — build a treap and race it against your AVL

Q10 claims randomisation gets the same guarantee for a fraction of the effort. Test that claim.

1. Implement a treap: random priority per node, BST invariant on keys, heap invariant on
   priorities, restored by the rotations §1.1 already gives you. Count the lines against §1.2 plus
   §1.3.
2. Verify it against a sorted set with **both** invariants asserted after every operation.
3. Feed it sorted keys and measure the height. It should land near AVL's, because the priorities are
   random even though the keys are not.
4. Measure insert and delete throughput against the AVL. Then write the paragraph you would put in
   a design doc recommending one over the other.

### Challenge 2 — implement red-black insert (and stop there)

§1.5 skipped the implementation. Do the insert half, which is tractable.

1. Implement red-black **insert** with its recolour-and-rotate cases. Leave delete out and say so,
   as this notebook did.
2. Write the invariant: all five properties, including equal black-height on every path — that last
   one is the interesting check and needs a recursive helper returning the black-height or a
   failure.
3. Stress it exactly as §1.3 stresses AVL, and print how often each case fires.
4. Measure the resulting heights against your AVL on the same keys, and confirm red-black comes out
   taller. Then measure rotations per insert for both.

### Challenge 3 — a B⁺-tree with range scans

§1.6 built a B-tree. The variant databases actually use is different in one important way.

1. Convert it to a **B⁺-tree**: all keys in the leaves, internal nodes hold only routing keys, and
   the leaves are **linked** left to right.
2. Implement `range(lo, hi)` as a descent to `lo` followed by a **sequential walk along the leaf
   links** — no tree traversal.
3. Measure node visits for range queries of varying width against the B-tree version from §1.6, and
   against NB-07 §2.4's pruning range query on a BST.
4. Explain, with your numbers, why `SELECT ... WHERE x BETWEEN a AND b` is the query B⁺-trees exist
   for.

***
# Part 5 - Practice

| # | Exercise | The technique | Difficulty |
|---|---|---|---|
| 1 | The four rotations, from scratch | Pointer surgery with height updates | ★★☆☆☆ |
| 2 | AVL insert | Detecting and dispatching the four cases | ★★★☆☆ |
| 3 | AVL delete | The cascade, and the balanced-child case | ★★★★☆ |
| 4 | Validate a balanced tree | Structural invariants, not behavioural | ★★★☆☆ |
| 5 | A treap | Randomisation instead of bookkeeping | ★★★☆☆ |
| 6 | Order-statistic tree | Augmentation surviving rotation | ★★★★☆ |
| 7 | B-tree insert with splitting | Growing upward from the root | ★★★★☆ |
| 8 | Interval tree | A different augmentation, a harder query | ★★★★★ |

***

### 1. The four rotations, from scratch

- **Brief:** `rotate_left` and `rotate_right`, plus the LR and RL compositions.
- **Good result:** a randomised test that rotating any node preserves the inorder sequence exactly
  (§1.1's test), over thousands of trees.
- **The trap:** **the order of the height updates.** The node that ends up lower must be updated
  first. §3 shows the wrong order produces a valid BST twice as tall as it should be — and that it
  looks fine on sorted input.

### 2. AVL insert

- **Brief:** insert as in a BST, then rebalance on the way back up.
- **Good result:** all four cases exercised (count them and print the counts), verified against a
  sorted set, with the invariant checked after every insert.
- **The trap:** deciding LL versus LR from the **child's** balance factor, not the grandchild's
  position — and remembering that a single rotation on a zig-zag just moves the imbalance.

### 3. AVL delete

- **Brief:** NB-07's three delete cases, plus rebalancing on the way up. Unlike insert, **do not
  stop at the first rebalance** — it can cascade to the root.
- **Good result:** thousands of randomised insert/delete sequences over a small key range, invariant
  after each.
- **The trap:** the case where the heavier child is **perfectly balanced**, which happens only on
  delete. The comparison must be `< 0` and not `<= 0`, or you take the double rotation when the
  single is correct. This is the bug that hand-written tests never reach.

### 4. Validate a balanced tree

- **Brief:** given a tree, decide whether it is a valid AVL tree — order, stored heights, balance
  factors.
- **Good result:** it rejects §3's three broken versions and accepts correct trees, on both sorted
  and shuffled input.
- **The trap:** checking `abs(height(left) - height(right)) <= 1` by *recomputing* the heights and
  never checking the **stored** ones. That version accepts `forgot_heights` — the tree is genuinely
  balanced by recomputation while the stored values are garbage, so the next insert misbehaves.
  Check both.

### 5. A treap

- **Brief:** random priority per node; BST on keys, max-heap on priorities; rotate to restore.
- **Good result:** verified against a sorted set with both invariants asserted; height on sorted
  input comparable to AVL's.
- **The trap:** deleting. Rotate the doomed node *down* until it is a leaf (always toward the
  higher-priority child), then detach it. Also: seed the RNG in tests, or failures are not
  reproducible — which matters more here than anywhere else in this notebook.

### 6. Order-statistic tree

- **Brief:** §2.2 — add `size` to each node, maintained through insert, delete and **every
  rotation**; implement `select(k)` and `rank(key)` in $O(h)$.
- **Good result:** both verified against a sorted list, with an invariant checking every node's
  stored size after every operation.
- **The trap:** the rotation update, in the wrong order, and the delete path where the key is
  copied from the successor but the size changes elsewhere. The invariant is the only thing that
  will tell you.

### 7. B-tree insert with splitting

- **Brief:** §1.6 — split a full child on the way down, promoting its median.
- **Good result:** the full invariant after every insert, including **equal leaf depth**, for
  several values of $t$.
- **The trap:** splitting **on the way down** rather than on the way back up. Doing it downward
  guarantees the parent always has room for the promoted key, which removes an entire class of
  cascade. Also: after splitting, re-decide which child to descend into — the key may now belong on
  the other side.

### 8. Interval tree

Store intervals; query for all intervals overlapping a given one.

- **Brief:** a BST keyed on interval start, augmented with the **maximum endpoint** in each subtree.
  Search prunes a subtree when its max endpoint is below the query's start.
- **Good result:** verified against brute force over thousands of random interval sets; the
  augmentation invariant checked after every operation, rotations included.
- **The trap:** the augmentation is `max(own end, left max, right max)` and must survive rotations —
  §2.2's lesson with a different summary. And the pruning argument is subtler than a range query's:
  work out *why* the max endpoint is the right thing to store before writing it.

***
# Part 6 - Reading

## Start here

**1. *Introduction to Algorithms* (CLRS), chapters 13 and 18.**
> Chapter 13 is red-black trees in full — every insert and delete case, with the loop invariants —
> which is exactly what §1.5 declines to reproduce and where to go if you want it. Chapter 18 is
> B-trees, including the split-on-the-way-down trick §1.6 uses and the delete cases §1.6 omits.
> Chapter 14, "Augmenting Data Structures", is §2.2 done rigorously and is short.

**2. [Sedgewick, *Left-Leaning Red-Black Trees*](https://sedgewick.io/wp-content/themes/sedgewick/papers/2008LLRB.pdf)** —
**Free.**
> Sedgewick's response to exactly the complaint §1.5 makes: red-black implementations are too long
> to write correctly. By restricting red links to lean left he gets insert and delete into a few
> dozen lines. Read it as a case study in *removing cases* — the same instinct as NB-05's sentinel
> nodes, applied to a much harder problem.

**3. [Seidel & Aragon, *Randomized Search Trees*](https://faculty.washington.edu/aragon/pubs/rst89.pdf)** —
Algorithmica 1996. **Free.**
> The treap. Q10's claim that randomisation buys the same guarantee for a fraction of the code, made
> properly. The key idea — assign random priorities so the tree behaves as though the keys arrived
> in random order — is the same move NB-03 §3 makes when it randomises a hash seed, and seeing it
> twice in different contexts is what makes it stick.

## The source behind each section

| Section | Where it comes from | Free? |
|---|---|---|
| 1.1 — rotations | **CLRS ch. 13.2** | 🔍 |
| 1.2–1.3 — AVL | **Adelson-Velsky & Landis**, 1962 — the original, and the first self-balancing tree | 🔍 |
| 1.4 — the height bound | **Knuth, TAOCP vol. 3 §6.2.3** — Fibonacci trees and the $1.44$ constant | 🔍 |
| 1.5 — red-black | **CLRS ch. 13**; **Guibas & Sedgewick**, *A dichromatic framework for balanced trees*, FOCS 1978 | 🔍 |
| 1.5 — the simplified variant | **Sedgewick**, *Left-Leaning Red-Black Trees*, 2008 | ✅ |
| 1.6 — **B-trees** | **Bayer & McCreight**, *Organization and maintenance of large ordered indices*, 1972; **CLRS ch. 18** | 🔍 |
| 1.6 — why databases care | **Graefe**, *Modern B-Tree Techniques*, 2011 — the survey | 🔍 |
| 2.2 — augmentation | **CLRS ch. 14** — order-statistic and interval trees | 🔍 |
| 2.3 — `TreeMap` | the JDK's `TreeMap` source, which is CLRS chapter 13 transcribed | ✅ |
| Q10 — treaps | **Seidel & Aragon**, 1996 | ✅ |
| Q10 — skip lists | **Pugh**, *Skip lists: a probabilistic alternative to balanced trees*, CACM 1990 | 🔍 |
| Q10 — splay trees | **Sleator & Tarjan**, *Self-adjusting binary search trees*, JACM 1985 | 🔍 |

**Legend:** ✅ free at the link · 🔍 search the exact title on
[Google Scholar](https://scholar.google.com)

### If you read only one

**Seidel & Aragon on treaps**, and read it against §3.

§3 measures how easy it is to get rotation code subtly wrong in a way that still passes every
behavioural test. The treap's answer is not "be more careful" — it is to **remove the thing that is
hard to get right**. There is no balance metadata, no case analysis, no LL/LR/RR/RL: assign each
node a random priority, maintain the heap property on those priorities, and the tree is shaped as
though the keys had arrived in random order regardless of how they actually arrived.

That is the same move as NB-03 §3's randomised hash seed and NB-07 §3.3's shuffle-before-insert, and
seeing it a third time is the point: **when the worst case depends on the input's order, randomness
you control can substitute for structure you would otherwise have to maintain.** It costs you a
worst-case guarantee in exchange for an expected one — which is exactly the trade Q12 says a
balanced tree exists to avoid, so the two papers are best read as a pair.

Then read **CLRS 13** if you want the red-black cases §1.5 skipped, and **CLRS 18** for B-trees.

***
# Appendix

| Symptom | Cause | Fix |
|---|---|---|
| Tree is correct but too tall | A rotation bug: still a valid BST, no longer balanced (§3) | Assert stored heights and balance factors, not behaviour |
| Heights are wrong after a rotation | Updated the new root before the node that moved below it (§1.1) | Update bottom-up: lower node first |
| Balance factor reaches ±2 and stays | Never doing the LR/RL double rotation (§3) | Dispatch on the **child's** balance factor |
| Tests pass on sorted input, production is slow | Sorted input never triggers LR/RL (§3) | Test with shuffled input too |
| Delete leaves the tree unbalanced | Stopped at the first rebalance (§1.3) | Rebalance at **every** level on the way up |
| Delete takes the wrong rotation | Heavier child perfectly balanced — only happens on delete (§1.2) | `< 0`, not `<= 0` |
| Augmented sizes drift wrong | Summary not maintained through rotations (§2.2) | Update it in the rotation, and check it in the invariant |
| `RecursionError` on a balanced tree | Depth is bounded — so this is a cycle, not depth (Part 0) | Check the invariant; you have corrupted the structure |
| B-tree leaves at different depths | Splitting on the way back up instead of down (§1.6) | Split the full child before descending into it |
| B-tree is slower than a BST in memory | Large $t$ makes each node insert $O(t)$ (§1.6) | B-trees are for paged storage; use a BST in memory |
| `TreeMap` slower than expected | Ordering costs ~10× against `HashMap` (§2.3) | If no ordered operation is used, use `HashMap` |
| Bulk load from sorted data is slow | Inserting one at a time does $n$ rebalances (§2.1) | Build from the median recursively — $\Theta(n)$, no rotations |

## Checklist for balanced-tree code

- [ ] Does the invariant check **stored** heights, not just recomputed ones (§3, Practice 4)?
- [ ] Is every balance factor asserted to be in $\{-1, 0, 1\}$ after **every** operation (§1.3)?
- [ ] Are the height updates in a rotation done **bottom-up** (§1.1)?
- [ ] Does the test print **how often each rotation case fired** (§1.3, Q11)?
- [ ] Does it test **shuffled** input, not only sorted (§3)?
- [ ] Does delete rebalance at every level, not just the first (§1.3)?
- [ ] Is any augmentation maintained **inside the rotation** and checked in the invariant (§2.2)?
- [ ] For sorted bulk input, is the tree built from the median rather than inserted (§2.1)?
- [ ] Is an ordered structure actually needed, or would a hash table do (§2.3)?
- [ ] Would the **library** do — is a hand-rolled tree justified by an augmentation (§2.4)?

## Where to go next

| Notebook | Why it follows |
|---|---|
| `heaps_zero_to_hero.ipynb` | A weaker tree invariant — partial order, in a flat array — when you only need the extreme |
| `tries_zero_to_hero.ipynb` | Trees keyed on the *structure* of the key rather than comparisons |
| `range_queries_zero_to_hero.ipynb` | §2.2's augmentation idea taken to its conclusion: Fenwick and segment trees |
| [`bst_zero_to_hero.ipynb`](bst_zero_to_hero.ipynb) | The problem this notebook solves, and the invariant it adds to |

See [`README.md`](README.md) for the full roster and reading order.